# Strut-and-Tie by Topology Optimization

Go from a **generic shape** (a rectangle or an arbitrary polygon, with supports
and loads) to an *optimized* strut-and-tie truss, automatically. This
complements the manual `StrutAndTieModel` workflow in *Strut and Tie Demo.ipynb*:
there you draw the truss; here civilpy **discovers** it — the goal being to make
the University of Toledo / ODOT strut-and-tie spreadsheet obsolete.

Pipeline (see `docs/StrutAndTieSolver.md`): mesh the region → SIMP topology
optimization → skeletonize the density for **node positions** → **LP plastic
truss layout optimization** for the discrete load path → size ties and check
nodes (AASHTO 5.8.2) → cost.

The layout optimization is the key step: it solves a single linear program for
the minimum-volume set of struts and ties in equilibrium with the loads. Because
an LP returns the *global* optimum, a symmetric problem gives a symmetric truss
— no path-dependent, symmetry-breaking greedy pruning.

## Workflow 1 — manual: a multi-column pier cap

A reinforced-concrete pier cap is the textbook D-region: stocky, with the loads
applied a short distance from the supports, so plane sections do **not** stay
plane and ordinary beam theory does not apply. Here is a realistic highway
layout — a 40 ft × 12 ft cap, 4 ft thick, on **three columns** (two 15 ft
spans with 5 ft overhangs), carrying **five girder reactions** of 250 kip across
the top.

Drawing this truss by hand is exactly what the spreadsheet automates. Civilpy
instead *discovers* it: the optimizer finds three compression fans down to the
columns, a tension chord across the top following the girders, and a diagonal web
tying it together — a genuine multi-panel strut-and-tie model, not a single
triangle. *Geometry carries what is spatial; tags carry what is scalar.*

In [ ]:
from civilpy.structural.stm_topology import DRegionProblem, Material

cap = DRegionProblem.rectangle(40, 12, thickness=4.0,
                               material=Material(f_c=5.0), vol_frac=0.30)
# three columns at the base — center pinned, outer columns on rollers
cap.add_support(5,  0, fix_x=False, fix_y=True, bearing=3.0)
cap.add_support(20, 0, fix_x=True,  fix_y=True, bearing=3.0)
cap.add_support(35, 0, fix_x=False, fix_y=True, bearing=3.0)
# five girder reactions across the top: overhang tips, mid-spans, over center
for x in (2.5, 12.5, 20, 27.5, 37.5):
    cap.add_load(x, 12, fy=-250, bearing=1.5)

result = cap.solve(nelx=140)   # mesh → SIMP → extract → refine → solve → size → cost
print(result.summary())

In [ ]:
# SIMP density (grey) with the discovered strut-and-tie truss overlaid:
# blue dashed = struts (compression) fanning to the columns,
# red = ties (tension) — the top chord following the girder loads.
result.plot();

In [ ]:
# result.model is an ordinary StrutAndTieModel — the solved forces feed the
# AASHTO capacity checks unchanged. A determinate result (DoI=0) means the
# refinement found a clean, statically-determinate load path.
for member, force in sorted(result.forces.items(), key=lambda kv: kv[1]):
    kind = 'tie ' if force > 0 else 'strut'
    print(f'{member[0]:>2}-{member[1]:<2}  {force:+8.1f} kip  ({kind})')

In [ ]:
# Tie reinforcement (AASHTO 5.8.2.4) and the ODOT cost take-off.
for t in result.report.ties:
    print(f'{t.member}: {t.force:6.1f} kip → {t.bar_count} x #{t.bar_size} '
          f'(As={t.a_st_provided:.2f} in^2, ratio={t.check.ratio:.2f})')
c = result.report.cost
print(f'\nTake-off: {c.concrete_cy:.1f} cy concrete + {c.steel_lb:.0f} lb steel '
      f'= ${c.total:,.0f}')

## Arbitrary polygons, voids, and keep-in solids

The optimizer takes any closed polygon, plus optional `voids` (mandatory holes)
and `solids` (keep-in zones). Here a duct void forces the load path around it.

In [ ]:
pv = DRegionProblem.rectangle(20, 10, thickness=2.0, vol_frac=0.4)
pv.voids = [[(8, 4), (12, 4), (12, 6), (8, 6)]]   # a duct opening
pv.add_support(1, 0, bearing=1.5)
pv.add_support(19, 0, fix_x=False, bearing=1.5)
pv.add_load(10, 10, fy=-600, bearing=1.5)
rv = pv.solve(nelx=100)
print(rv.summary())
rv.plot();

## Workflow 2 — Rhino: draw the region, optimize here

Instead of `add_*` calls, draw the concrete outline in Rhino as a closed curve
tagged `stm.kind=region` (with `stm.thickness`, `stm.fc`), tag the supports and
loads exactly as in the drawn-truss workflow, and read it in one call. The tag
schema is the frozen contract in `docs/Rhino Design Philosophy.md`. Needs the
optional `rhino3dm` dependency (`pip install civilpy[rhino]`).

In [ ]:
# from civilpy.structural.stm_topology import DRegionProblem
#
# problem = DRegionProblem.from_3dm('PierCap_Region.3dm')  # stm.kind=region + supports/loads
# result = problem.solve()
# result.plot()
#
# # round-trip the discovered truss back to Rhino for review
# from civilpy.structural import rhino_stm
# rhino_stm.results_to_3dm(result.model, 'PierCap_STM_results.3dm')

## Under the hood

Each stage is independently usable:

```python
from civilpy.structural.stm_topology import (
    GroundMesh, optimize_density, extract_truss, layout_optimize_truss)
mesh    = GroundMesh(cap, nelx=140)                  # Phase 1  (structured quads)
density = optimize_density(mesh, vol_frac=0.30)      # Phase 2  (SIMP)
skel    = extract_truss(density)                     # Phase 3  (skeleton joints)
model   = layout_optimize_truss(skel, density)       # Phase 4  (LP layout opt.)
forces  = model.forces                               # already solved
```

**Phase 4 — plastic truss layout optimization** is what makes a busy layout
work. Skeleton tracing only fixes the joint *positions* (cleaned by clustering,
and mirrored when the problem is symmetric). The discrete load path is then
chosen by a linear program that minimizes the Michell structural volume
`sum |Fᵢ|·Lᵢ·wᵢ` subject to nodal equilibrium `B·F = P`, over a ground structure
of candidate members. Each member splits into a tension part and a compression
part, so ties and struts share one linear model; `wᵢ` is a gentle density weight
(members down dense bands are cheaper, via a line integral of the SIMP field),
candidates crossing a void are filtered out, and a tolerance betweenness filter
keeps chords as a single clean run. The LP's global optimum is inherently
symmetric, and the surviving non-zero-force members *are* the strut-and-tie
model — no greedy pruning. The fully-stressed ground-structure refinement
(`refine_truss(..., method='fsd')`) remains as a fallback for the rare
infeasible LP.

The indeterminate truss solver also stands alone on any `StrutAndTieModel` via
`solve(method='auto')`.